In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import fmatoolbox as fma
import isautilities as isau
import xarray as xr
import powerlaw
import warnings

import pathlib
froot = pathlib.Path().cwd().parent.parent / 'Results' / 'Figures' / 'ISAHpcPfc'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
do_save = False

In [ ]:
def _powerlaw_fit(data, xmin=None, xmax=None):
    """Fit a discrete power law via Clauset et al. 2009 (MLE).
    returns dict with: tau, sigma, xmin, xmax, R_vs_exp, p_vs_exp, n_tail_proportion, ks_stat, r2 (all NaN on failure)"""
    data = np.asarray(data).astype(int)
    data = data[data >= 1]
    data_unique, counts = np.unique(data, return_counts=True)
    print(len(data), "avalanches for power-law fit", len(data_unique), "unique values")
    result = {key: np.nan for key in ('tau','sigma','xmin','xmax','r','p','tail_prop','ks','r2')}

    # validate input
    if len(data_unique) < 2 or len(data) < 20:
        print("DATA TOO SMALL OR NOT ENOUGH UNIQUE VALUES FOR POWER-LAW FIT")
        return result
    most_common_freq = np.max(counts) / len(data)
    if most_common_freq > 0.90:
        print("MOST COMMON VALUE TOO PREVALENT FOR POWER-LAW FIT")
        return result

    if xmax is not None:
        xmax = int(np.percentile(data,xmax)) # apply percentile

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", OptimizeWarning)
        warnings.simplefilter("ignore", UserWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        try:
            fit = powerlaw.Fit(data, xmin=xmin, xmax=xmax, discrete=True, verbose=False)
        except Exception:
            return result
    if np.isnan(fit.power_law.alpha):
        return result

    result['tau'] = fit.power_law.alpha
    result['sigma'] = fit.power_law.sigma
    result['xmin'] = fit.xmin
    result['xmax'] = fit.xmax if fit.xmax is not None else np.max(data_unique[-1])
    result['tail_prop'] = fit.n_tail / len(data)

    # compare power law against exponential
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result['r'], result['p'] = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)

        tail = data[(data >= xmin) & (data <= xmax)]
        theoretical = fit.power_law.cdf(np.sort(tail))
        empirical   = np.arange(1,len(tail)+1) / len(tail)
        result['ks'] = float(np.max(np.abs(empirical - theoretical)))

        # R² en log-log sur la queue fittée
        x_ccdf, y_ccdf = _ccdf(tail)
        log_x = np.log(x_ccdf)
        log_y = np.log(y_ccdf + 1e-12)
        valid = np.isfinite(log_x) & np.isfinite(log_y)
        if valid.sum() > 2:
            y_pred = np.polyval(np.polyfit(log_x[valid], log_y[valid], 1), log_x[valid])
            ss_res = np.sum((log_y[valid] - y_pred)**2)
            ss_tot = np.sum((log_y[valid] - log_y[valid].mean())**2)
            result['r2'] = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        else:
            result['r2'] = np.nan

    return result


def _ccdf(data: np.ndarray):
    """Compute empirical CCDF P(X >= x)."""
    data = np.sort(data)
    n    = len(data)
    ccdf = 1.0 - np.arange(1, n + 1) / n   # P(X >= x)  (right-continuous)
    return data, ccdf


def _z_test_tau(tau_a, sigma_a, tau_b, sigma_b):
    """
    Two-sided Z-test H0: tau_A == tau_B.
    sigma_a/b are standard errors from MLE.
    Returns (z, p_two_sided).
    """
    from scipy.stats import norm
    se = np.sqrt(sigma_a**2 + sigma_b**2)
    if se == 0 or np.isnan(se):
        return np.nan, np.nan
    z = (tau_a - tau_b) / se
    p = 2 * norm.sf(np.abs(z))
    return float(z), float(p)


def _gamma_from_scaling(sizes: np.ndarray, durations: np.ndarray) -> float:
    """Estimate gamma via OLS regression of log<S|D> on log D.
    Returns slope (= gamma), or NaN."""
    sizes     = np.asarray(sizes,     dtype=float)
    durations = np.asarray(durations, dtype=float)
    valid = (sizes >= 1) & (durations >= 1)
    S, D  = sizes[valid], durations[valid]

    unique_D = np.unique(D)
    if len(unique_D) < 4:
        return np.nan

    mean_S = np.array([S[D == d].mean() for d in unique_D])
    log_D  = np.log(unique_D)
    log_S  = np.log(mean_S)

    keep = np.isfinite(log_D) & np.isfinite(log_S)
    if keep.sum() < 4:
        return np.nan

    gamma, _ = np.polyfit(log_D[keep], log_S[keep], 1)
    return float(gamma)


class AvalancheComparison:
    """
    High-level API for comparing avalanche statistics across brain states.

    Parameters
    ----------
    session       : neuroscience session object understood by rg.data.Regions
    bin_size      : float  — bin size (s) passed to get_avalanches
    threshold     : float  — threshold passed to get_avalanches
    get_avalanches: callable(FR, bin_size, threshold) -> (sizes, durations)
                    If None, the module will look for a globally available
                    `get_avalanches` function.
    """

    # ── public run methods ────────────────────────────────────────────────────

    def run_sws_vs_other(self, region: str = "nr", xmin= None, xmax= None):
        """
        Compare SWS avalanches vs non-SWS (other) avalanches.
        """

        # R, then fr in states, then aval, then exps
        # missing stats!
        self.stats = _compare_two(*self.results)
        return self

    # ── plotting ──────────────────────────────────────────────────────────────

    def plot(self, figsize=(16, 10)) -> plt.Figure:
        """
        Generate the full comparison figure.

        Layout
        ------
        Row 0 :  CCDF size (A)  |  CCDF duration (B)
        Row 1 :  Metrics bar chart (C)   |   Stats table (D)
        """
        if not self.results:
            raise RuntimeError("Call run_*() before plot().")

        fig = plt.figure(figsize=figsize, constrained_layout=True)
        fig.suptitle(self._title, fontsize=14, fontweight="bold", y=1.01)

        gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
        ax_size  = fig.add_subplot(gs[0, 0])
        ax_dur   = fig.add_subplot(gs[0, 1])
        ax_met   = fig.add_subplot(gs[1, 0])
        ax_stat  = fig.add_subplot(gs[1, 1])

        self._plot_ccdf(ax_size, which="size")
        self._plot_ccdf(ax_dur,  which="duration")
        self._plot_metrics(ax_met)
        self._plot_stats_table(ax_stat)

        return fig

    # ── internal plot helpers ─────────────────────────────────────────────────

    def _plot_ccdf(self, ax: plt.Axes, which: str = "size"):
        """Log-log CCDF + power-law fit for sizes or durations."""
        is_size = (which == "size")
        for res in self.results:
            raw  = res.sizes     if is_size else res.durations
            tau  = res.tau_size  if is_size else res.tau_dur
            xmin = res.xmin_size if is_size else res.xmin_dur
            xmax = res.xmax_size if is_size else res.xmax_dur

            raw = raw[np.isfinite(raw) & (raw >= 1)]
            if len(raw) < 5:
                continue

            x_ccdf, y_ccdf = _ccdf(raw)
            ax.plot(x_ccdf, y_ccdf, ".", color=res.color,
                    alpha=0.4, markersize=3, rasterized=True)

            # ── power-law line in the fitted range ──────────────────────
            if np.isfinite(tau) and np.isfinite(xmin):
                xs  = np.logspace(np.log10(xmin), np.log10(xmax), 200)
                # normalise so that the line passes through the CCDF at xmin
                idx   = np.searchsorted(x_ccdf, xmin)
                y0    = y_ccdf[min(idx, len(y_ccdf) - 1)]
                slope = -(tau - 1)
                ys    = y0 * (xs / xmin) ** slope
                R_val = res.R_size if is_size else res.R_dur
                p_val = res.p_size if is_size else res.p_dur
                ks_val = res.ks_size  if is_size else res.ks_dur
                r2_val = res.r2_size  if is_size else res.r2_dur

                label = (f"{res.label}  τ={tau:.2f}±{(res.sigma_size if is_size else res.sigma_dur):.2f}"
                         f"\n[xmin={xmin:.0f}, xmax={xmax:.0f}, xmin/xmax={xmin/xmax:.2f}, n_tail={res.ntail_size:.1%}]"
                         f"\n[R={R_val:.2f}, p_vs_exp={p_val:.3f}, KS={ks_val:.3f}, R²={r2_val:.3f}]")
                ax.axvline(xmin, color=res.color, linewidth=0.8, linestyle=':', alpha=0.7)
                ax.axvline(xmax, color=res.color, linewidth=0.8, linestyle='--', alpha=0.5)
                ax.plot(xs, ys, "-", color=res.color, linewidth=2, label=label)

        xlabel = "Avalanche size $s$" if is_size else "Avalanche duration $d$"
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel(r"$P(S \geq s)$" if is_size else r"$P(D \geq d)$", fontsize=11)
        ax.set_title("Size CCDF" if is_size else "Duration CCDF", fontsize=12)
        ax.legend(fontsize=9, framealpha=0.7)
        ax.grid(True, which="both", alpha=0.25, linestyle="--")

    def _plot_metrics(self, ax):
        """Grouped bar chart of criticality metrics per condition."""
        metrics = [
            ("γ emp.\n(τ_dur-1)/(τ_size-1)", "gamma_empirical"),
            ("γ scaling\n⟨S⟩~D^γ",            "tau_gamma_scaling"),
            ("DCC\n|γ_emp - γ_scal|",          "DCC"),
        ]
        n_metrics = len(metrics)
        n_cond    = len(self.results)
        width     = 0.8 / n_cond
        x         = np.arange(n_metrics)

        for i, res in enumerate(self.results):
            vals = [getattr(res, attr) for _, attr in metrics]
            offset = (i - (n_cond - 1) / 2) * width
            bars = ax.bar(x + offset, vals, width * 0.9,
                          color=res.color, label=res.label, alpha=0.85,
                          edgecolor="white", linewidth=0.5)
            for bar, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.01,
                            f"{v:.3f}", ha="center", va="bottom", fontsize=7.5)

        ax.set_xticks(x)
        ax.set_xticklabels([m for m, _ in metrics], fontsize=9.5)
        ax.set_ylabel("Value", fontsize=11)
        ax.set_title("Criticality metrics", fontsize=12)
        ax.legend(fontsize=9, framealpha=0.7)
        ax.axhline(0, color="black", linewidth=0.6)
        ax.grid(axis="y", alpha=0.25, linestyle="--")

    def _plot_stats_table(self, ax):
        """Table summarising all statistical tests."""
        ax.axis("off")
        if self.stats is None:
            return

        s  = self.stats
        la = self.results[0].label
        lb = self.results[1].label

        def _fmt(v):
            return f"{v:.4f}" if np.isfinite(v) else "—"

        def _star(p):
            if not np.isfinite(p): return ""
            if p < 0.001: return "***"
            if p < 0.01:  return "**"
            if p < 0.05:  return "*"
            return "n.s."

        rows = [
            ["Test", "Statistic", "p-value", "Sig."],
            # ── Z-tests ──────────────────────────────────────────────
            [f"Z-test τ_size\n({la} vs {lb})",
             f"z={_fmt(s.z_tau_size)}", _fmt(s.p_z_tau_size), _star(s.p_z_tau_size)],
            [f"Z-test τ_dur\n({la} vs {lb})",
             f"z={_fmt(s.z_tau_dur)}", _fmt(s.p_z_tau_dur), _star(s.p_z_tau_dur)],
            # ── KS tests ─────────────────────────────────────────────
            [f"KS sizes\n({la} vs {lb})",
             f"D={_fmt(s.ks_size)}", _fmt(s.p_ks_size), _star(s.p_ks_size)],
            [f"KS durations\n({la} vs {lb})",
             f"D={_fmt(s.ks_dur)}", _fmt(s.p_ks_dur), _star(s.p_ks_dur)],
            # ── metric diffs ─────────────────────────────────────────
            [f"Δγ empirical\n({la}−{lb})",
             f"{_fmt(s.delta_gamma)}", "—", ""],
            [f"Δγ scaling\n({la}−{lb})",
             f"{_fmt(s.delta_tau_scaling)}", "—", ""],
            [f"ΔDCC\n({la}−{lb})",
             f"{_fmt(s.delta_DCC)}", "—", ""],
        ]

        col_widths = [0.38, 0.24, 0.20, 0.10]
        n_rows     = len(rows)
        row_h      = 1.0 / n_rows

        for r_idx, row in enumerate(rows):
            y = 1.0 - (r_idx + 0.5) * row_h
            is_header = (r_idx == 0)
            x_cursor  = 0.0

            # row background
            bg = "#e8e8e8" if is_header else ("#f7f7f7" if r_idx % 2 == 0 else "white")
            rect = plt.Rectangle((0, 1.0 - (r_idx + 1) * row_h),
                                  1.0, row_h,
                                  transform=ax.transAxes,
                                  color=bg, zorder=0, clip_on=False)
            ax.add_patch(rect)

            for c_idx, (cell, cw) in enumerate(zip(row, col_widths)):
                xc = x_cursor + cw / 2
                fw = "bold" if is_header else "normal"
                fs = 8.5 if is_header else 8

                # colour p-value column based on significance
                fc = "black"
                if c_idx == 2 and not is_header:
                    try:
                        pv = float(cell)
                        if pv < 0.001:   fc = "#006d2c"
                        elif pv < 0.01:  fc = "#31a354"
                        elif pv < 0.05:  fc = "#74c476"
                        else:            fc = "#d73027"
                    except ValueError:
                        pass

                ax.text(xc, y, cell, ha="center", va="center",
                        fontsize=fs, fontweight=fw, color=fc,
                        transform=ax.transAxes, wrap=True,
                        multialignment="center")
                x_cursor += cw

        # outer border
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title("Statistical comparison", fontsize=12, pad=6)

    # ── convenience: print summary ────────────────────────────────────────────

    def summary(self):
        """Print a text summary of all metrics and comparisons."""
        print(f"\n{'='*60}")
        print(f"  {self._title}")
        print(f"{'='*60}")
        for res in self.results:
            print(f"\n  ── {res.label} (n={len(res.sizes)}) ──")
            print(f"     τ_size          = {res.tau_size:.4f} ± {res.sigma_size:.4f}"
                  f"   (xmin={res.xmin_size}, p_vs_exp={res.p_size:.3f})")
            print(f"     τ_dur           = {res.tau_dur:.4f} ± {res.sigma_dur:.4f}"
                  f"   (xmin={res.xmin_dur}, p_vs_exp={res.p_dur:.3f})")
            print(f"     γ empirical     = {res.gamma_empirical:.4f}")
            print(f"     γ scaling       = {res.tau_gamma_scaling:.4f}")
            print(f"     DCC             = {res.DCC:.4f}")

        if self.stats:
            s = self.stats
            la, lb = self.results[0].label, self.results[1].label
            print(f"\n  ── Statistical tests ({la} vs {lb}) ──")
            print(f"     Z τ_size  z={s.z_tau_size:.3f}  p={s.p_z_tau_size:.4f}")
            print(f"     Z τ_dur   z={s.z_tau_dur:.3f}  p={s.p_z_tau_dur:.4f}")
            print(f"     KS sizes  D={s.ks_size:.3f}   p={s.p_ks_size:.4f}")
            print(f"     KS durs   D={s.ks_dur:.3f}    p={s.p_ks_dur:.4f}")
        print()


In [ ]:
def _critExp(session,regs=None,when='sleep.*#0',states=[['slownr'],['sws','/slownr']],xmin=None,xmax=None):

    R = fma.regions.regions(session,phases=when,states=['sws','rem'],events='InfraSlowRhythm/infraslowaval')
    regs = R.ids if regs is None else np.array(regs)[np.isin(regs,R.ids)]
    sizes, intervals, _ = R.avalanches(regs=regs,thresh=30,window=0.05)

    fit = np.full((len(regs),len(states),2,8), np.nan) # 4th dim is: sigma, xmin, xmax, R, p, tail_prop, ks, r2
    exponents = np.full((len(regs),len(states),5), np.nan) # 3rd dim is: tau_s, tau_d, gamma_th, gamma_emp, dcc
    for i, r in enumerate(regs):
        for j, state in enumerate(states):
            _, valid = fma.general.restrict(intervals[r][:,0], R.eventIntervals(state), s_ind=True)
            sizes_this = sizes[r][valid]
            durations = np.diff(intervals[r][valid]).ravel()

            # sizes
            size_fit = _powerlaw_fit(sizes_this, xmin=xmin, xmax=xmax)
            fit[i,j,0] = (size_fit['sigma'],size_fit['xmin'],size_fit['xmax'],size_fit['r'],size_fit['p'],size_fit['tail_prop'],size_fit['ks'],size_fit['r2'])
            exponents[i,j,0] = size_fit['tau']

            # durations
            dur_fit = _powerlaw_fit(durations, xmin=xmin, xmax=xmax)
            fit[i,j,1] = (dur_fit['sigma'],dur_fit['xmin'],dur_fit['xmax'],dur_fit['r'],dur_fit['p'],dur_fit['tail_prop'],dur_fit['ks'],dur_fit['r2'])
            exponents[i,j,1] = dur_fit['tau']

            # DCC
            exponents[i,j,2] = (dur_fit['tau'] - 1) / (size_fit['tau'] - 1) # theoretical gamma from tau exponents
            exponents[i,j,3] = _gamma_from_scaling(sizes_this, durations) # gamma from <S> ~ D^gamma scaling relation
            exponents[i,j,4] = abs(exponents[i,j,2] - exponents[i,j,3]) # DCC
    states = (sum(s) for s in states)
    fit = xr.DataArray(fit,dis=('reg','state','q','val'),coords={'reg': regs, 'state': states, 'q': ('S','D'),
                                                                 'val': ('sigma','xmin','xmax','R','p','tail_prop','ks','r2')})
    exponents = xr.DataArray(exponents,dis=('reg','state','val'),coords={'reg': regs, 'state': states, 'val': ('tau_s','tau_d','gamma_th','gamma_sc','DCC')})

    return fit, exponents

In [ ]:
# test on one session
session = fma.data.readBatchFile(batch_file)[0][15]
print(session)
fit, exponents = _critExp(session)